# 07 — Network Optimization (Sequential Greedy)

**Sequential greedy station placement.**  
Identifies AFIR coverage gaps on interurban roads. Places new stations at high-demand gap
midpoints (or nearest service area) using sequential scoring: `V_i = n_chargers_needed × gap_length_km`.

Inputs: interurban roads + existing charger baseline + ABM demand per segment  
Output: `proposed_stations.csv`

## Data Inputs
- `data/processed/interurban_roads.parquet` — road network with geometry
- `data/processed/interurban_chargers_baseline.csv` — existing chargers
- `data/processed/demand_per_segment.csv` — ABM demand from NB06
- `data/processed/service_areas_clean.geojson` — preferred candidate sites

## Data Output
- `data/processed/proposed_stations.csv`
  - Columns: `location_id, latitude, longitude, route_segment, n_chargers_proposed`

In [2]:
import os
import sys
import pandas as pd
import geopandas as gpd
import numpy as np
from pathlib import Path

if os.path.basename(os.getcwd()) == 'notebooks':
    sys.path.insert(0, os.path.dirname(os.getcwd()))
    DATA_DIR = Path('../data/processed')
else:
    sys.path.insert(0, os.getcwd())
    DATA_DIR = Path('data/processed')

from src.constants import (
    MAX_STATION_SPACING_KM,
    MAX_STATION_SPACING_TENT_CORE_KM,
    MAX_STATION_SPACING_TENT_COMP_KM,
    MIN_EXISTING_CHARGER_POWER_KW,
    MIN_CHARGERS_STANDARD, MIN_CHARGERS_TENT,
)
from src.optimization import compute_coverage_gaps, place_stations_greedy
from src.data_loading import load_geo_parquet_compat

print('✅ Imports OK')
print(f'   AFIR spacing — TEN-T Core:   {MAX_STATION_SPACING_TENT_CORE_KM} km')
print(f'   AFIR spacing — TEN-T Comp:   {MAX_STATION_SPACING_TENT_COMP_KM} km')
print(f'   AFIR spacing — General:      {MAX_STATION_SPACING_KM} km')
print(f'   Min existing power for coverage: {MIN_EXISTING_CHARGER_POWER_KW} kW')

✅ Imports OK
   AFIR spacing — TEN-T Core:   60 km
   AFIR spacing — TEN-T Comp:   100 km
   AFIR spacing — General:      120 km
   Min existing power for coverage: 50 kW


## Step 1: Load inputs

In [3]:
# Road network
roads = load_geo_parquet_compat(DATA_DIR / 'interurban_roads.parquet')
print(f'📊 Roads: {len(roads):,} segments')

# Derive tent_tier from TENT_red_basica (interurban_roads.parquet stores the raw NB03 column,
# not the mapped tier). Without this, compute_coverage_gaps() falls back to treating ALL
# TEN-T routes as Core (60 km), incorrectly over-tightening Comprehensive corridors.
def _tent_tier(row):
    val = row.get('TENT_red_basica')
    if isinstance(val, str):
        v = val.strip().lower()
        if v == 'core':
            return 'core'
        if v == 'comprehensive':
            return 'comprehensive'
    return 'core' if row.get('is_tent', False) else 'none'

roads['tent_tier'] = roads.apply(_tent_tier, axis=1)
print(f'   TEN-T tiers — Core: {(roads["tent_tier"]=="core").sum()}, '
      f'Comprehensive: {(roads["tent_tier"]=="comprehensive").sum()}, '
      f'None: {(roads["tent_tier"]=="none").sum()}')

# Existing chargers baseline
chargers = pd.read_csv(DATA_DIR / 'interurban_chargers_baseline.csv')
print(f'🔌 Existing chargers (all): {len(chargers):,}')
fast_chargers = chargers[chargers['max_power_kw'] >= MIN_EXISTING_CHARGER_POWER_KW]
print(f'   Fast chargers (≥{MIN_EXISTING_CHARGER_POWER_KW} kW, valid coverage): {len(fast_chargers):,}')

# ABM demand output
demand_path = DATA_DIR / 'demand_per_segment.csv'
if demand_path.exists():
    demand = pd.read_csv(demand_path)
    print(f'📈 Demand segments: {len(demand):,}')
    print(f'   n_chargers range: {demand["n_chargers_needed"].min()} – {demand["n_chargers_needed"].max()}')
else:
    demand = pd.DataFrame(columns=['segment_id', 'daily_bev_traffic_2027', 'n_chargers_needed', 'is_tent'])
    print(f'⚠️  Missing {demand_path.name}; continuing with empty demand frame until NB06 is run')

# Service areas (preferred candidate locations)
sa_path = DATA_DIR / 'service_areas_clean.geojson'
service_areas = gpd.read_file(sa_path) if sa_path.exists() else None
sa_count = len(service_areas) if service_areas is not None else 0
print(f'🅿️  Service areas available: {sa_count}')

📊 Roads: 1,295 segments
   TEN-T tiers — Core: 183, Comprehensive: 202, None: 910
🔌 Existing chargers (all): 6,065
   Fast chargers (≥50 kW, valid coverage): 3,246
📈 Demand segments: 1,295
   n_chargers range: 2 – 12
🅿️  Service areas available: 113


## Step 2: Identify AFIR Coverage Gaps

For each road segment, find nearest fast charger (≥50 kW).
Flag segments where distance > AFIR threshold (60 / 100 / 120 km by road type).

In [4]:
gaps = compute_coverage_gaps(
    road_segments_df=roads,
    existing_stations_df=chargers,
)

print(f'🔍 Coverage gap analysis (road-following linear referencing):')
print(f'   Total routes:       {roads["Carretera"].nunique():,}')
print(f'   Routes with gaps:   {gaps["Carretera"].nunique():,}')
print(f'   Total gap stretches: {len(gaps):,}')

if len(gaps) > 0 and 'gap_spacing_threshold_km' in gaps.columns:
    labels = {60: 'TEN-T Core (60 km)', 100: 'TEN-T Comp (100 km)', 120: 'General (120 km)'}
    for thresh, count in gaps['gap_spacing_threshold_km'].value_counts().sort_index().items():
        print(f'   {labels.get(thresh, f"{thresh} km")}: {count:,} gap stretches')
    print(f'\n   Longest gaps:')
    print(
        gaps.nlargest(10, 'gap_length_km')[
            ['Carretera', 'gap_start_km', 'gap_end_km', 'gap_length_km', 'is_tent', 'gap_spacing_threshold_km']
        ].to_string(index=False)
    )

🔍 Coverage gap analysis (road-following linear referencing):
   Total routes:       281
   Routes with gaps:   9
   Total gap stretches: 9
   TEN-T Core (60 km): 4 gap stretches
   TEN-T Comp (100 km): 3 gap stretches
   General (120 km): 2 gap stretches

   Longest gaps:
Carretera  gap_start_km  gap_end_km  gap_length_km  is_tent  gap_spacing_threshold_km
    N-435         48.35      197.63         149.27    False                       120
    N-621          0.00      129.23         129.23    False                       120
    N-322         28.94      155.96         127.02     True                       100
    N-433          0.00      120.01         120.01     True                       100
    N-330          1.73      112.40         110.67     True                       100
     AP-2          6.58      109.59         103.01     True                        60
      N-2         19.43      103.39          83.96     True                        60
     A-23         92.27      169.46    

## Step 3: Sequential Greedy Placement

Score: `V_i = n_chargers_needed × gap_length_km`  
Select top candidate, mark covered road segments, repeat until all gaps closed.  
Prefer service area locations when within 5 km of gap midpoint.

In [5]:
proposed = place_stations_greedy(
    gap_segments_df=gaps,
    demand_df=demand,
    service_areas_gdf=service_areas,
)

print(f'🏗️  Sequential greedy placement complete:')
print(f'   Proposed new stations: {len(proposed):,}')

if len(proposed) > 0:
    print(f'\n   Charger distribution per station:')
    print(proposed['n_chargers_proposed'].value_counts().sort_index().to_string())
    total_chargers = proposed['n_chargers_proposed'].sum()
    print(f'\n   Total chargers: {total_chargers:,}  |  Total capacity: {total_chargers*150:,} kW')

🏗️  Sequential greedy placement complete:
   Proposed new stations: 8

   Charger distribution per station:
n_chargers_proposed
2    2
4    6

   Total chargers: 28  |  Total capacity: 4,200 kW


/Users/theo/Code/iberdrola-ev-network/src/optimization.py:378: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  sa_pts['_sa_lat'] = sa_pts.geometry.centroid.y
/Users/theo/Code/iberdrola-ev-network/src/optimization.py:379: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  sa_pts['_sa_lon'] = sa_pts.geometry.centroid.x


## Step 4: Validation & Save

In [6]:
if len(proposed) == 0:
    print('ℹ️  No gaps — creating empty proposed_stations.csv')
    proposed = pd.DataFrame(columns=[
        'location_id', 'latitude', 'longitude', 'route_segment', 'n_chargers_proposed'
    ])
else:
    required = ['location_id', 'latitude', 'longitude', 'route_segment', 'n_chargers_proposed']
    missing = set(required) - set(proposed.columns)
    assert not missing, f'Missing columns: {missing}'
    assert proposed['latitude'].between(35.0, 44.5).all(), 'Latitude out of Spain bounds'
    assert proposed['longitude'].between(-10.0, 5.0).all(), 'Longitude out of Spain bounds'
    assert proposed['n_chargers_proposed'].between(2, 12).all(), 'Charger count out of range'
    assert proposed['location_id'].is_unique, 'Duplicate location_ids'
    print(f'✅ Validation passed: {len(proposed):,} stations, all checks OK')

out_path = DATA_DIR / 'proposed_stations.csv'
proposed[['location_id', 'latitude', 'longitude', 'route_segment', 'n_chargers_proposed']].to_csv(
    out_path, index=False
)
print(f'💾 Saved → {out_path}')
proposed.head()

✅ Validation passed: 8 stations, all checks OK
💾 Saved → ../data/processed/proposed_stations.csv


,location_id,latitude,longitude,route_segment,n_chargers_proposed
0,STA_0001,38.775545,-2.446657,N-322,4
1,STA_0002,37.900207,-6.644477,N-433,4
2,STA_0003,39.935600,-1.297858,N-330,4
3,STA_0004,41.512416,-0.068422,AP-2,4
4,STA_0005,41.495475,-0.183176,N-2,4
